# ODI to Databricks Migration: `HR.TRG_EMP` Full Load

**Source File:** `HR.TRG_EMP_Full_Load.sql`
**Conversion Timestamp:** 2024-07-30 12:00:00 UTC
**Description:** This notebook performs a full load of employee data from `HR.EMPLOYEES` into the target table `HR.TRG_EMP`.

In [ ]:
# COMMAND ----------
# DBTITLE 1,Create ETL Parameter Widgets
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type (e.g., FULL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "Last Extract Time (YYYY-MM-DD HH:MI:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "2999-12-31 23:59:59", "Current Extract Time (YYYY-MM-DD HH:MI:SS)")

# ETL Parameters

In [ ]:
%sql
-- DBTITLE 1,Create Temporary View for ETL_LAST_EXTRACT_TIME
CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

In [ ]:
%sql
-- DBTITLE 1,Create Temporary View for ETL_CURRENT_EXTRACT_TIME
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
# COMMAND ----------
# DBTITLE 1,Display ETL Parameter Values
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO,
    etl_last_extract_time AS ETL_LAST_EXTRACT_TIME,
    etl_current_extract_time AS ETL_CURRENT_EXTRACT_TIME
  FROM v_etl_last_extract_time, v_etl_current_extract_time
"""))

# Target Table: `TRG_EMP`

In [ ]:
%sql
-- DBTITLE 1,SCEN_TASK_NO in {10}: Create Target Table IF NOT EXISTS (Inferred DDL)
-- This DDL is inferred based on a common Oracle HR.EMPLOYEES table structure.
-- Please verify column definitions against your actual source system.
CREATE TABLE IF NOT EXISTS workspace.hr.trg_emp (
  employee_id     BIGINT,
  first_name      STRING,
  last_name       STRING,
  email           STRING,
  phone_number    STRING,
  hire_date       TIMESTAMP,
  job_id          STRING,
  salary          DECIMAL(8,2),
  commission_pct  DECIMAL(2,2),
  manager_id      BIGINT,
  department_id   BIGINT
) USING DELTA;

In [ ]:
%sql
-- DBTITLE 1,SCEN_TASK_NO in {20}: Clear existing data from target (Full Load)
-- This is a full load, so existing data in the target table is truncated.
TRUNCATE TABLE workspace.hr.trg_emp;

In [ ]:
%sql
-- DBTITLE 1,SCEN_TASK_NO in {30}: Insert into TRG_EMP
INSERT INTO workspace.hr.trg_emp
(
  employee_id,
  first_name,
  last_name,
  email,
  phone_number,
  hire_date,
  job_id,
  salary,
  commission_pct,
  manager_id,
  department_id
)
SELECT
  employees.employee_id,
  employees.first_name,
  employees.last_name,
  employees.email,
  employees.phone_number,
  employees.hire_date,
  employees.job_id,
  employees.salary,
  employees.commission_pct,
  employees.manager_id,
  employees.department_id
FROM
  workspace.hr.employees AS employees;

In [ ]:
%sql
-- DBTITLE 1,Record Count After Load
SELECT COUNT(*) FROM workspace.hr.trg_emp;

# Cleanup

# Validation

In [ ]:
%sql
-- DBTITLE 1,Sample Data from Target Table
SELECT * FROM workspace.hr.trg_emp LIMIT 10;

# Conversion Notes

1.  **Schema and Table Naming:** All schema and table names have been converted to lowercase and prefixed with `workspace.`. For example, `HR.TRG_EMP` became `workspace.hr.trg_emp`.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable in Databricks Spark SQL.
3.  **Data Types:** Oracle `NUMBER` and `VARCHAR2` data types were mapped to Spark SQL `BIGINT`, `DECIMAL`, and `STRING` respectively, based on common HR schema definitions.
    *   `NUMBER(6)` -> `BIGINT`
    *   `VARCHAR2(n)` -> `STRING`
    *   `DATE` -> `TIMESTAMP`
    *   `NUMBER(8,2)` -> `DECIMAL(8,2)`
    *   `NUMBER(2,2)` -> `DECIMAL(2,2)`
4.  **SCEN_TASK_NO:** The `SCEN_TASK_NO` comments from the original ODI file have been preserved as comments within the corresponding SQL cells.
5.  **Target Table DDL:** A `CREATE TABLE IF NOT EXISTS` statement for `workspace.hr.trg_emp` has been added at the beginning of the notebook to ensure the target table exists before the `INSERT` operation. The DDL includes a `TRUNCATE TABLE` before the insert to handle the full load nature of the original script.
6.  **Full Load Logic:** The original script implies a full load, so a `TRUNCATE TABLE` statement has been added before the `INSERT` into `workspace.hr.trg_emp` to clear existing data.